In [1]:
import numpy as np

file_read = "006_refit_noise"
file_write = "007_RMSE_noise"

In [2]:
column_target = "beta_av"
column_target_scaled = "beta_av_logit"
column_pred = f"{column_target}_pred"
column_pred_scaled = f"{column_target_scaled}_pred"

In [3]:
import polars as pl

dfs_rmse_min = []

for ratio in np.arange(0, 1.1, 0.1):
    
    if ratio == 0.0:
        df_preds = pl.read_parquet("001_refit_preds.parquet")
    else:
        df_preds = pl.read_parquet(f"{file_read}_preds_{ratio:.1f}.parquet")
    
    df_rmse = (
        df_preds
        .with_columns(
            (100 / (1 + (-pl.col(column_target_scaled)).exp())).alias(column_target),
            (100 / (1 + (-pl.col(column_pred_scaled)).exp())).alias(column_pred),
        )
        .group_by("n_features", maintain_order=True)
        .agg(
            (pl.col(column_target) - pl.col(column_pred)).pow(2).mean().sqrt()
            .alias(f"{column_target}_rmse")
        )
    )
    df_rmse_min = (
        df_rmse
        .filter(
            pl.col(f"{column_target}_rmse") == pl.col(f"{column_target}_rmse").min()
        )
        .with_columns(pl.lit(ratio).alias("noise_ratio"))
        .select(["noise_ratio", "n_features", f"{column_target}_rmse"])
    )
    dfs_rmse_min.append(df_rmse_min)

In [4]:
dfs_rmse_min = pl.concat(dfs_rmse_min)

In [5]:
dfs_rmse_min

noise_ratio,n_features,beta_av_rmse
f64,i32,f64
0.0,11,3.861018
0.1,11,3.884787
0.2,11,4.14617
0.3,10,4.617747
0.4,10,5.195546
…,…,…
0.6,11,6.600627
0.7,23,7.514426
0.8,21,8.249834


In [6]:
dfs_rmse_min.write_parquet(f"{file_write}_summary.parquet")